# Neural Texture Compression (NTC) — Colab Experiment (`PNTC` Custom Scheme)

PyTorch experiment demonstrating the core ideas of Neural Texture Compression:
- **Multi-scale 2D latent feature grids** sampled with bilinear interpolation (`grid_sample`)
- **Quantization-aware training** via Straight-Through Estimator (STE, 4-bit / 8-bit latents)
- **Compact MLP decoder** jointly reconstructing single high-res textures (`sponza`) or multi-channel PBR stacks (`emergence`: Albedo + Normal + Specular)
- **Custom `.pntc` binary export & round-trip decode verification** (independent of NVIDIA `libntc`)

> **Runtime**: `Runtime -> Change runtime type -> T4 GPU` before running.

In [ ]:
!nvidia-smi
import torch
print(f"PyTorch: {torch.__version__}, CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Device: {torch.cuda.get_device_name(0)}")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [ ]:
import os
import math
import struct
import numpy as np
from PIL import Image
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt

# SOURCE_MODE:
#   'upload'    - upload one or more textures directly in Colab (e.g. from projects/demos/sponza or projects/demos/emergence)
#   'repo'      - load from a cloned prisma repo (set REPO_ROOT below)
#   'synthetic' - generate a procedural PBR-like test stack if no files are provided
SOURCE_MODE = "upload"
PRESET = "sponza"  # 'sponza' or 'emergence' (used when SOURCE_MODE == 'repo')
REPO_ROOT = "/content/prisma"

# Optional resize to keep T4 experiment fast (set None to keep native resolution)
MAX_SIDE = 1024


def load_image_tensor(path, max_side=None):
    img = Image.open(path).convert("RGB")
    if max_side is not None and max(img.size) > max_side:
        scale = max_side / max(img.size)
        new_size = (max(1, int(img.width * scale)), max(1, int(img.height * scale)))
        img = img.resize(new_size, Image.Resampling.LANCZOS)
    arr = np.asarray(img, dtype=np.float32) / 255.0
    return torch.from_numpy(arr).permute(2, 0, 1)  # (3, H, W)


def make_synthetic_pbr(size=512):
    y, x = torch.meshgrid(
        torch.linspace(-1, 1, size),
        torch.linspace(-1, 1, size),
        indexing="ij",
    )
    r = torch.sqrt(x**2 + y**2)
    albedo = torch.stack([
        0.5 + 0.4 * torch.sin(12 * x) * torch.cos(12 * y),
        0.4 + 0.3 * torch.cos(8 * r),
        0.3 + 0.3 * torch.sin(20 * (x + y)),
    ], dim=0).clamp(0, 1)
    normal = torch.stack([
        0.5 + 0.35 * torch.cos(12 * x),
        0.5 + 0.35 * torch.sin(12 * y),
        torch.full_like(x, 0.9),
    ], dim=0).clamp(0, 1)
    return [("synthetic_albedo", albedo), ("synthetic_normal", normal)]


textures = []
if SOURCE_MODE == "upload":
    try:
        from google.colab import files
        uploaded = files.upload()
        for name in sorted(uploaded.keys()):
            textures.append((name, load_image_tensor(name, MAX_SIDE)))
    except ImportError:
        print("Not running in Colab; falling back to synthetic test stack.")
elif SOURCE_MODE == "repo":
    if PRESET == "sponza":
        candidates = [
            os.path.join(REPO_ROOT, "projects/demos/sponza/assets/models/sponza/14118779221266351425.jpg"),
        ]
    else:
        base = os.path.join(REPO_ROOT, "projects/demos/emergence/assets/textures/blocks")
        candidates = [
            os.path.join(base, "stone_bricks.png"),
            os.path.join(base, "stone_bricks_n.png"),
            os.path.join(base, "stone_bricks_s.png"),
        ]
    for p in candidates:
        if os.path.exists(p):
            textures.append((os.path.basename(p), load_image_tensor(p, MAX_SIDE)))
        else:
            print(f"Missing: {p}")

if not textures:
    print("Using synthetic fallback textures.")
    textures = make_synthetic_pbr(512)

# Align all textures in the stack to the first texture's spatial size
target_h, target_w = textures[0][1].shape[1:]
aligned = []
for name, t in textures:
    if t.shape[1:] != (target_h, target_w):
        t = F.interpolate(t.unsqueeze(0), size=(target_h, target_w), mode="bilinear", align_corners=False).squeeze(0)
    aligned.append((name, t))
textures = aligned

tex_names = [n for n, _ in textures]
target_stack = torch.cat([t for _, t in textures], dim=0).to(device)  # (C_total, H, W)
C_total, H, W = target_stack.shape
print(f"Loaded {len(textures)} texture(s): {tex_names}, shape=({C_total}, {H}, {W})")

In [ ]:
# NTC compression hyperparameters
STRIDE_HI = 4       # High-res latent grid downscale factor
STRIDE_LO = 16      # Low-res latent grid downscale factor
CH_HI = 8           # Channels per texel in high-res latent grid
CH_LO = 8           # Channels per texel in low-res latent grid
LATENT_BITS = 8     # Simulated quantization bit-width (4 or 8)
QUANT_BOUND = 1.0   # Symmetric quantization clamp bound [-QUANT_BOUND, +QUANT_BOUND]
HIDDEN_DIM = 64     # Compact MLP hidden width
NUM_HIDDEN = 2      # Number of hidden layers


def quantize_ste(x, bits=8, bound=1.0):
    # Symmetric uniform quantization with Straight-Through Estimator
    if bits >= 16:
        return x
    qmax = (1 << (bits - 1)) - 1
    x_clamped = x.clamp(-bound, bound)
    scale = bound / qmax
    x_q = torch.round(x_clamped / scale) * scale
    return x_clamped + (x_q - x_clamped).detach()


class NeuralTexture(nn.Module):
    def __init__(self, h, w, out_channels, stride_hi=4, stride_lo=16, ch_hi=8, ch_lo=8, hidden=64, num_hidden=2, bits=8, bound=1.0):
        super().__init__()
        self.h = h
        self.w = w
        self.out_channels = out_channels
        self.bits = bits
        self.bound = bound
        self.hidden = hidden
        self.num_hidden = num_hidden

        h_hi, w_hi = max(2, h // stride_hi), max(2, w // stride_hi)
        h_lo, w_lo = max(2, h // stride_lo), max(2, w // stride_lo)

        self.grid_hi = nn.Parameter(torch.empty(1, ch_hi, h_hi, w_hi).uniform_(-0.05, 0.05))
        self.grid_lo = nn.Parameter(torch.empty(1, ch_lo, h_lo, w_lo).uniform_(-0.05, 0.05))

        in_dim = ch_hi + ch_lo
        layers = [nn.Linear(in_dim, hidden), nn.GELU()]
        for _ in range(num_hidden - 1):
            layers.extend([nn.Linear(hidden, hidden), nn.GELU()])
        layers.extend([nn.Linear(hidden, out_channels), nn.Sigmoid()])
        self.mlp = nn.Sequential(*layers)

    def forward(self, coords):
        # coords: (N, 2) in [-1, 1], ordered (x, y) for grid_sample
        sample_grid = coords.view(1, 1, -1, 2)
        q_hi = quantize_ste(self.grid_hi, self.bits, self.bound)
        q_lo = quantize_ste(self.grid_lo, self.bits, self.bound)

        f_hi = F.grid_sample(q_hi, sample_grid, mode="bilinear", padding_mode="border", align_corners=False)
        f_lo = F.grid_sample(q_lo, sample_grid, mode="bilinear", padding_mode="border", align_corners=False)
        feat = torch.cat([f_hi, f_lo], dim=1).squeeze(0).squeeze(1).t()  # (N, ch_hi + ch_lo)
        return self.mlp(feat)

    def compression_stats(self, h, w, raw_channels):
        latent_elems = self.grid_hi.numel() + self.grid_lo.numel()
        mlp_params = sum(p.numel() for p in self.mlp.parameters())
        total_bits = latent_elems * self.bits + mlp_params * 16
        bpp = total_bits / (h * w)
        raw_bpp = raw_channels * 8.0
        return {
            "latent_kb": (latent_elems * self.bits) / 8 / 1024,
            "mlp_kb": (mlp_params * 16) / 8 / 1024,
            "bpp": bpp,
            "raw_bpp": raw_bpp,
            "ratio": raw_bpp / bpp,
        }


model = NeuralTexture(
    H, W, C_total,
    stride_hi=STRIDE_HI, stride_lo=STRIDE_LO,
    ch_hi=CH_HI, ch_lo=CH_LO,
    hidden=HIDDEN_DIM, num_hidden=NUM_HIDDEN,
    bits=LATENT_BITS, bound=QUANT_BOUND,
).to(device)

stats = model.compression_stats(H, W, C_total)
print(f"NTC budget: {stats['bpp']:.3f} bpp (vs raw {stats['raw_bpp']:.1f} bpp -> {stats['ratio']:.2f}x compression)")
print(f"Latents: {stats['latent_kb']:.1f} KB | MLP (FP16): {stats['mlp_kb']:.1f} KB")

In [ ]:
# Training configuration
STEPS = 3000
BATCH_PIXELS = 65536
LR = 1e-2

optimizer = torch.optim.Adam(model.parameters(), lr=LR)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=STEPS, eta_min=1e-4)
scaler = torch.amp.GradScaler("cuda", enabled=(device.type == "cuda"))

loss_history = []
psnr_history = []

model.train()
for step in range(1, STEPS + 1):
    # Continuous sub-pixel sampling in [-1, 1] helps reduce grid aliasing
    coords = torch.rand(BATCH_PIXELS, 2, device=device) * 2.0 - 1.0
    gt = F.grid_sample(
        target_stack.unsqueeze(0),
        coords.view(1, 1, -1, 2),
        mode="bilinear",
        padding_mode="border",
        align_corners=False,
    ).squeeze(0).squeeze(1).t()

    optimizer.zero_grad(set_to_none=True)
    with torch.amp.autocast("cuda", enabled=(device.type == "cuda")):
        pred = model(coords)
        loss = F.mse_loss(pred, gt)

    scaler.scale(loss).backward()
    scaler.step(optimizer)
    scaler.update()
    scheduler.step()

    loss_val = loss.item()
    psnr_val = -10.0 * math.log10(max(loss_val, 1e-10))
    loss_history.append(loss_val)
    psnr_history.append(psnr_val)

    if step % 300 == 0 or step == 1:
        print(f"Step {step:4d}/{STEPS} | MSE: {loss_val:.6f} | PSNR: {psnr_val:.2f} dB | LR: {scheduler.get_last_lr()[0]:.2e}")

In [ ]:
@torch.no_grad()
def reconstruct_full_image(model, h, w, chunk_size=131072):
    model.eval()
    ys = (torch.arange(h, device=device, dtype=torch.float32) + 0.5) / h * 2.0 - 1.0
    xs = (torch.arange(w, device=device, dtype=torch.float32) + 0.5) / w * 2.0 - 1.0
    gy, gx = torch.meshgrid(ys, xs, indexing="ij")
    coords = torch.stack([gx, gy], dim=-1).reshape(-1, 2)

    preds = []
    for i in range(0, coords.shape[0], chunk_size):
        preds.append(model(coords[i : i + chunk_size]).cpu())
    out = torch.cat(preds, dim=0).reshape(h, w, -1).permute(2, 0, 1)
    return out.clamp(0.0, 1.0)


recon_stack = reconstruct_full_image(model, H, W)
gt_cpu = target_stack.cpu()

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(loss_history)
axes[0].set_yscale("log")
axes[0].set_title("Training MSE")
axes[0].set_xlabel("Step")
axes[1].plot(psnr_history)
axes[1].set_title("Training PSNR (dB)")
axes[1].set_xlabel("Step")
plt.tight_layout()
plt.show()

num_tex = len(textures)
fig, axes = plt.subplots(num_tex, 3, figsize=(15, 5 * num_tex), squeeze=False)
for idx, name in enumerate(tex_names):
    c0, c1 = idx * 3, (idx + 1) * 3
    gt_img = gt_cpu[c0:c1].permute(1, 2, 0).numpy()
    rc_img = recon_stack[c0:c1].permute(1, 2, 0).numpy()
    mse = float(np.mean((gt_img - rc_img) ** 2))
    psnr = -10.0 * math.log10(max(mse, 1e-10))
    err = np.clip(np.abs(gt_img - rc_img).mean(axis=-1) * 10.0, 0.0, 1.0)

    axes[idx, 0].imshow(gt_img)
    axes[idx, 0].set_title(f"Original: {name}")
    axes[idx, 0].axis("off")

    axes[idx, 1].imshow(rc_img)
    axes[idx, 1].set_title(f"NTC ({stats['bpp']:.2f} bpp) | PSNR: {psnr:.2f} dB")
    axes[idx, 1].axis("off")

    axes[idx, 2].imshow(err, cmap="inferno")
    axes[idx, 2].set_title("10x Abs Error Heatmap")
    axes[idx, 2].axis("off")

plt.tight_layout()
plt.show()

## Custom Binary Scheme: `PNTC` v1 (`.pntc`)

This notebook uses a **custom, self-contained binary format** (`PNTC`, Prisma Neural Texture) rather than NVIDIA's `.ntc` container so both training and Slang/Rust GPU decoding are 100% customizable.

### Little-Endian Binary Layout
| Offset | Field | Type | Description |
| :--- | :--- | :--- | :--- |
| `0x00` | `magic` | `char[4]` | `b"PNTC"` |
| `0x04` | `version` | `u16` | `1` |
| `0x06` | `flags` | `u16` | `0` (GELU hidden layers + Sigmoid output) |
| `0x08` | `width` | `u32` | Target texture width `W` |
| `0x0C` | `height` | `u32` | Target texture height `H` |
| `0x10` | `out_channels` | `u16` | Total reconstructed channels `C_total` |
| `0x12` | `latent_bits` | `u8` | `8` (`int8`) or `4` (2 signed 4-bit nibbles per `uint8`) |
| `0x13` | `num_hidden` | `u8` | Number of MLP hidden layers |
| `0x14` | `hi_h`, `hi_w`, `ch_hi` | `u16[3]` | High-res latent grid shape |
| `0x1A` | `lo_h`, `lo_w`, `ch_lo` | `u16[3]` | Low-res latent grid shape |
| `0x20` | `hidden_dim` | `u16` | MLP hidden layer width |
| `0x22` | `bound` | `f32` | Symmetric quantization bound (`q_float = q_int * (bound / qmax)`) |
| `0x26` | `grid_hi` | `bytes` | Quantized `(ch_hi, hi_h, hi_w)` C-contiguous payload |
| `...` | `grid_lo` | `bytes` | Quantized `(ch_lo, lo_h, lo_w)` C-contiguous payload |
| `...` | `mlp_layers` | `f16[]` | For each Linear layer: `weight[out, in]` (`f16`) then `bias[out]` (`f16`) |

In [ ]:
PNTC_HEADER_FMT = "<4sHHIIHBBHHHHHHHf"  # 38 bytes
PNTC_MAGIC = b"PNTC"
PNTC_VERSION = 1


def pack_quantized_grid(tensor, bits, bound):
    qmax = (1 << (bits - 1)) - 1
    scale = bound / qmax
    q_int = torch.round(tensor.detach().cpu().clamp(-bound, bound) / scale).to(torch.int8).numpy().ravel()
    if bits == 8:
        return q_int.tobytes()
    if bits == 4:
        # Offset [-7, 7] to [0, 14] and pack 2 nibbles per uint8 (low nibble first)
        u = (q_int.astype(np.int16) + qmax).astype(np.uint8)
        if u.size % 2 != 0:
            u = np.append(u, np.uint8(qmax))
        packed = (u[0::2] & 0x0F) | ((u[1::2] & 0x0F) << 4)
        return packed.tobytes()
    raise ValueError(f"Unsupported latent_bits={bits}; expected 4 or 8")


def unpack_quantized_grid(raw_bytes, shape, bits, bound):
    qmax = (1 << (bits - 1)) - 1
    scale = bound / qmax
    numel = int(np.prod(shape))
    if bits == 8:
        q_int = np.frombuffer(raw_bytes, dtype=np.int8, count=numel).astype(np.float32)
    elif bits == 4:
        packed = np.frombuffer(raw_bytes, dtype=np.uint8)
        u = np.empty(packed.size * 2, dtype=np.int16)
        u[0::2] = packed & 0x0F
        u[1::2] = (packed >> 4) & 0x0F
        q_int = (u[:numel] - qmax).astype(np.float32)
    else:
        raise ValueError(f"Unsupported latent_bits={bits}")
    return torch.from_numpy((q_int * scale).reshape(shape))


def export_pntc(model, path):
    _, ch_hi, hi_h, hi_w = model.grid_hi.shape
    _, ch_lo, lo_h, lo_w = model.grid_lo.shape
    header = struct.pack(
        PNTC_HEADER_FMT,
        PNTC_MAGIC,
        PNTC_VERSION,
        0,
        model.w,
        model.h,
        model.out_channels,
        model.bits,
        model.num_hidden,
        hi_h,
        hi_w,
        ch_hi,
        lo_h,
        lo_w,
        ch_lo,
        model.hidden,
        float(model.bound),
    )
    hi_bytes = pack_quantized_grid(model.grid_hi, model.bits, model.bound)
    lo_bytes = pack_quantized_grid(model.grid_lo, model.bits, model.bound)

    mlp_chunks = []
    for m in model.mlp:
        if isinstance(m, nn.Linear):
            mlp_chunks.append(m.weight.detach().cpu().to(torch.float16).numpy().tobytes())
            mlp_chunks.append(m.bias.detach().cpu().to(torch.float16).numpy().tobytes())

    with open(path, "wb") as f:
        f.write(header)
        f.write(hi_bytes)
        f.write(lo_bytes)
        for chunk in mlp_chunks:
            f.write(chunk)

    size_bytes = os.path.getsize(path)
    actual_bpp = (size_bytes * 8.0) / (model.h * model.w)
    print(f"Exported {path}: {size_bytes / 1024:.2f} KB ({actual_bpp:.3f} bpp actual file size)")
    return size_bytes


PNTC_PATH = "texture_compressed.pntc"
export_pntc(model, PNTC_PATH)

In [ ]:
@torch.no_grad()
def decode_pntc_file(path, chunk_size=131072):
    with open(path, "rb") as f:
        data = f.read()

    hdr_size = struct.calcsize(PNTC_HEADER_FMT)
    (
        magic, version, flags, w, h, out_ch, bits, num_hidden,
        hi_h, hi_w, ch_hi, lo_h, lo_w, ch_lo, hidden_dim, bound,
    ) = struct.unpack(PNTC_HEADER_FMT, data[:hdr_size])
    assert magic == PNTC_MAGIC and version == PNTC_VERSION, f"Invalid PNTC header: {magic}, v{version}"

    def grid_byte_len(numel):
        return numel if bits == 8 else (numel + 1) // 2

    offset = hdr_size
    hi_len = grid_byte_len(ch_hi * hi_h * hi_w)
    grid_hi = unpack_quantized_grid(data[offset : offset + hi_len], (1, ch_hi, hi_h, hi_w), bits, bound).to(device)
    offset += hi_len

    lo_len = grid_byte_len(ch_lo * lo_h * lo_w)
    grid_lo = unpack_quantized_grid(data[offset : offset + lo_len], (1, ch_lo, lo_h, lo_w), bits, bound).to(device)
    offset += lo_len

    # Rebuild MLP layers from FP16 payload
    dims = [ch_hi + ch_lo] + [hidden_dim] * num_hidden + [out_ch]
    layers = []
    for idx in range(len(dims) - 1):
        in_d, out_d = dims[idx], dims[idx + 1]
        w_bytes = in_d * out_d * 2
        b_bytes = out_d * 2
        w_arr = np.frombuffer(data[offset : offset + w_bytes], dtype=np.float16).reshape(out_d, in_d).astype(np.float32)
        offset += w_bytes
        b_arr = np.frombuffer(data[offset : offset + b_bytes], dtype=np.float16).astype(np.float32)
        offset += b_bytes

        lin = nn.Linear(in_d, out_d).to(device)
        lin.weight.copy_(torch.from_numpy(w_arr))
        lin.bias.copy_(torch.from_numpy(b_arr))
        layers.append(lin)
        layers.append(nn.GELU() if idx < len(dims) - 2 else nn.Sigmoid())

    assert offset == len(data), f"Trailing bytes in PNTC file: {len(data) - offset}"
    decoder_mlp = nn.Sequential(*layers).eval()

    ys = (torch.arange(h, device=device, dtype=torch.float32) + 0.5) / h * 2.0 - 1.0
    xs = (torch.arange(w, device=device, dtype=torch.float32) + 0.5) / w * 2.0 - 1.0
    gy, gx = torch.meshgrid(ys, xs, indexing="ij")
    coords = torch.stack([gx, gy], dim=-1).reshape(-1, 2)

    preds = []
    for i in range(0, coords.shape[0], chunk_size):
        sg = coords[i : i + chunk_size].view(1, 1, -1, 2)
        f_hi = F.grid_sample(grid_hi, sg, mode="bilinear", padding_mode="border", align_corners=False)
        f_lo = F.grid_sample(grid_lo, sg, mode="bilinear", padding_mode="border", align_corners=False)
        feat = torch.cat([f_hi, f_lo], dim=1).squeeze(0).squeeze(1).t()
        preds.append(decoder_mlp(feat).cpu())

    return torch.cat(preds, dim=0).reshape(h, w, out_ch).permute(2, 0, 1).clamp(0.0, 1.0)


decoded_from_file = decode_pntc_file(PNTC_PATH)
roundtrip_diff = float((decoded_from_file - recon_stack).abs().max())
file_mse = float(((decoded_from_file - gt_cpu) ** 2).mean())
file_psnr = -10.0 * math.log10(max(file_mse, 1e-10))
print(f"Round-trip max abs diff (vs in-memory FP32 MLP): {roundtrip_diff:.6f}")
print(f"Decoded .pntc overall PSNR: {file_psnr:.2f} dB")